In [26]:
import yaml
import numpy as np
import os
from tqdm import tqdm
import sys
sys.path.append("src")
from satpos import *
from sppp import *
from ptk_yaml import *
from lambda_common import *

In [27]:
#测试多系统PPP-AR
def UCPPP_M(obs_mats,obs_start,obs_epoch,IGS,clk,
          sat_out,ion_param,sat_pcos,el_threthod,ex_threshold_v,ex_threshold_v_sigma,Mw_threshold,GF_threshold,dy_mode,
          X,Pk,Qk,phase_bias,X_time,GF_sign,Mw_sign,slip_sign,dN_sign,sat_num,out_age,freqs,AMB_FIX=0,sta_mode='None',RTK_Info={},AMB_FIX_Info={}):
    
    Out_log,Out_log_fix=[],[]

    obs_index=obs_start
    for epoch in tqdm(range(obs_epoch)):
    
        #观测时间
        rt_week=obs_mats[0][obs_index+epoch][0]['GPSweek']
        rt_sec=obs_mats[0][obs_index+epoch][0]['GPSsec']
        rt_unix=satpos.gpst2time(rt_week,rt_sec)
    
        #进行单点定位
        spp_rr,rnx_obs,peph_sat_pos=SPP_from_IGS_M(obs_mats,obs_index+epoch,IGS,clk,sat_out,ion_param,sat_pcos,freqs,el_threthod=el_threthod,sol_mode="IF",pre_rr=[X[0][0],X[1][0],X[2][0],X[3][0]])
        #无单点定位解
        if(not len(rnx_obs)):
            print("No valid observations, Pass epoch: Week: {}, sec: {}.".format(rt_week,rt_sec))
            continue
        #流动站重收敛
        if(RTK_Info['reinitial_sec'] and rt_sec%(RTK_Info['reinitial_sec'])==0):
            # 分别初始化各系统PPP子滤波状态与协方差
            X_G,Pk_G,Qk_G,GF_sign_G,Mw_sign_G,slip_sign_G,dN_sign_G,X_time_G,phase_bias_G=init_UCPPP(obs_mats[0],obs_index+epoch,IGS,clk,sat_out,ion_param,sat_pcos,sys_sat_num=32,f1=freqs[0][0],f2=freqs[0][1])
            X_C,Pk_C,Qk_C,GF_sign_C,Mw_sign_C,slip_sign_C,dN_sign_C,X_time_C,phase_bias_C=init_UCPPP(obs_mats[1],obs_index+epoch,IGS,clk,sat_out,ion_param,sat_pcos,sys_sat_num=65,f1=freqs[1][0],f2=freqs[1][1])
            X_E,Pk_E,Qk_E,GF_sign_E,Mw_sign_E,slip_sign_E,dN_sign_E,X_time_E,phase_bias_E=init_UCPPP(obs_mats[2],obs_index+epoch,IGS,clk,sat_out,ion_param,sat_pcos,sys_sat_num=37,f1=freqs[2][0],f2=freqs[2][1])
    
            #多系统非差PPP滤波状态初始化
            X,Pk,Qk,GF_sign,Mw_sign,slip_sign,dN_sign,X_time,phase_bias=init_UCPPP_M(X_G,X_C,X_E,
                 Pk_G,Pk_C,Pk_E,
                 Qk_G,Qk_C,Qk_E,
                 GF_sign_G,GF_sign_C,GF_sign_E,
                 Mw_sign_G,Mw_sign_C,Mw_sign_E,
                 slip_sign_G,slip_sign_C,slip_sign_E,
                 dN_sign_G,dN_sign_C,dN_sign_E,
                 X_time_G,X_time_C,X_time_E,
                 phase_bias_G,phase_bias_C,phase_bias_E)

        #PPP状态更新
        updata_PPP_state_M(X,Pk,spp_rr,epoch,rt_unix,X_time,Qk,GF_sign,Mw_sign,slip_sign,dN_sign,GF_threshold,Mw_threshold,sat_num,rnx_obs,out_age,freqs,dy_mode)

        #PPP时间更新
        X,Pk,Qk,X_time,v,phase_bias,rnx_obs=KF_UCPPP_M(X,X_time,Pk,Qk,ion_param,peph_sat_pos,rnx_obs,ex_threshold_v,ex_threshold_v_sigma,rt_unix,phase_bias,freqs,AMB_FIX,mode=sta_mode,RTK_Info=RTK_Info,AMB_FIX_Info=AMB_FIX_Info,Out_log_fix=Out_log_fix)

        #结果保存
        #Out_log.append([X[0][0],X[1][0],X[2][0]])
        Out_log.append(log2out_M(rt_unix,v,rnx_obs,X,X_time,Pk,peph_sat_pos,freqs))
    
    return Out_log,Out_log_fix

def find_elmax(rnx_obs,peph_sat_pos,Xk_dot,t_Pk,sys='G'):
    el_max=0.0
    el_max_id=999
    for i in range(len(rnx_obs)):
        si_PRN=rnx_obs[i]['PRN']
        if(si_PRN[0]!=sys):
            continue
        #ion_index=5+3*sys_shift+PRN_index
        N1_index=5+len(rnx_obs)+i
        N2_index=5+2*len(rnx_obs)+i
        #计算高度角
        sat_xyzt=peph_sat_pos[rnx_obs[i]['PRN']]
        #读取模糊度状态的方差
        tk_N1=t_Pk[N1_index][N1_index]
        tk_N2=t_Pk[N2_index][N2_index]
        _,el=getazel(sat_xyzt,[Xk_dot[0][0],Xk_dot[1][0],Xk_dot[2][0]])
        # if(tk_N1>2.25 or tk_N2>2.25):
        #     #模糊度状态在本历元重置, 不能作为参考星
        #     continue
        if(el>el_max):
            el_max=el
            el_max_id=i
    if(el_max_id!=999):
        el_max_prn=rnx_obs[el_max_id]['PRN']
    else:
        el_max_prn='X00'
    return el_max_prn,el_max_id


def PPP_AR_M(rnx_obs,peph_sat_pos,Xk_dot,t_Pk,freqs,ratio_threshold=2.0,amb_float_threshold=1.0,P_float_threshold=1.5**2,min_AR_num=3):
    #从Pk和X中恢复整数部分
    Xfloat_N12=np.zeros((2*len(rnx_obs),1))
    Pfloat_N12=np.zeros((2*len(rnx_obs),2*len(rnx_obs)))
    lambda_trans=np.zeros((2*len(rnx_obs),2*len(rnx_obs)))
    el_max_prns,el_max_ids=['X00','X00','X00'],[999,999,999]
    #逐系统查找高度角最高卫星
    el_max_prns[0],el_max_ids[0]=find_elmax(rnx_obs,peph_sat_pos,Xk_dot,t_Pk,'G')
    el_max_prns[1],el_max_ids[1]=find_elmax(rnx_obs,peph_sat_pos,Xk_dot,t_Pk,'C')
    el_max_prns[2],el_max_ids[2]=find_elmax(rnx_obs,peph_sat_pos,Xk_dot,t_Pk,'E')
    #星间单差算子
    for id in range(len(rnx_obs)):
        #卫星PRN
        si_prn=rnx_obs[id]['PRN']
        el_max_id=el_max_ids[['G','C','E'].index(si_prn[0])]#设置系统对应参考星
        f1,f2=freqs[['G','C','E'].index(si_prn[0])][0],freqs[['G','C','E'].index(si_prn[0])][1]
        #状态和方差信息获取
        Xfloat_N12[id][0]=Xk_dot[5+len(rnx_obs)+id][0]
        Pfloat_N12[id]=t_Pk[5+len(rnx_obs)+id][5+len(rnx_obs):-2]#注意模糊度状态范围
        Xfloat_N12[len(rnx_obs)+id]=Xk_dot[5+2*len(rnx_obs)+id][0]
        Pfloat_N12[len(rnx_obs)+id]=t_Pk[5+2*len(rnx_obs)+id][5+len(rnx_obs):-2]
        
        #星间单差算子
        lambda_trans[id][id]=f1/clight
        lambda_trans[len(rnx_obs)+id][len(rnx_obs)+id]=f2/clight
        lambda_trans[id][el_max_id]=-f1/clight
        lambda_trans[len(rnx_obs)+id][len(rnx_obs)+el_max_id]=-f2/clight#星间单差算子
    #星间单差
    Xfloat_N12=lambda_trans.dot(Xfloat_N12)
    Pfloat_N12=lambda_trans.dot(Pfloat_N12).dot(lambda_trans.T)
    #构建星间单差模糊度矩阵
    X_float_N12_SD=[]
    id_use=[]
    #首先循环选择标准差小于1.5周的单差模糊度
    for id in range(len(Xfloat_N12)):
        t=np.array(el_max_ids)+len(rnx_obs)
        if(id not in el_max_ids and id not in t and Pfloat_N12[id][id]<P_float_threshold):
            #大于0.15周的模糊度不固定
            N_float_part=abs(round(Xfloat_N12[id][0])-Xfloat_N12[id][0])
            if(N_float_part>amb_float_threshold):
                continue
            X_float_N12_SD.append(Xfloat_N12[id])
            id_use.append(id)
    #如果无候选模糊度或候选模糊度过少, 则部分模糊度固定失败, 返回失败标志
    if(len(X_float_N12_SD)<min_AR_num):
        return False
    P_float_N12_SD=[]
    for id in id_use:
        P_temp=[]
        for j_id in id_use:
            P_temp.append(Pfloat_N12[id][j_id])
        P_float_N12_SD.append(P_temp.copy())
    #如果位置内符合精度太差, 跳过模糊度固定
    # if(t_Pk[0][0]>0.25 or t_Pk[1][1]>0.25 or t_Pk[2][2]>0.25):
    #     return False
    X_float_N12_SD=np.array(X_float_N12_SD).reshape((len(X_float_N12_SD),1))
    try:
        #ratios,ds,N12_FIX=LAMBDA_FIX(Xfloat_N12,Pfloat_N12)
        ratios,ds,N12_FIX=LAMBDA_FIX(X_float_N12_SD,P_float_N12_SD,loopmax=50000)
        #N12_FIX_res=(N12_FIX-X_float_N12_SD.T)
    except:
        ratios=[0.0,0.0]
        ds=[9999,9999]
    if(ratios[0]>ratio_threshold or ratios[1]>ratio_threshold):
        #单次模糊度固定ration校验通过
        #返回模糊度固定信息
        N12_SD_FIX_info={'ratios':[ratios[0],ratios[1]]}
        N12_SD_FIX_info['Ref_sat']=el_max_prns
        N12_SD_FIX_info['N12_SD_sat']=[]
        N12_SD_FIX_info['N12_SD_value']=[]
        for id in id_use:
            #L1频率上的星间单差模糊度
            if (id<len(rnx_obs)):
                N12_SD_FIX_info['N12_SD_sat'].append([rnx_obs[id]['PRN'],'N1'])
            #L2频率上的星间单差模糊度
            else:
                N12_SD_FIX_info['N12_SD_sat'].append([rnx_obs[id-len(rnx_obs)]['PRN'],'N2'])
            N12_SD_FIX_info['N12_SD_value'].append(N12_FIX[id_use.index(id)])
        return N12_SD_FIX_info
    else:
        #单次模糊度校验不通过, 进行部分模糊度固定(PAR)
        #方法一、依次剔除方差较大的模糊度
        if(len(X_float_N12_SD)==min_AR_num):
            return False#最小模糊度固定数量固定失败, 返回False
        sub_id_use,t_ratios=PAR_Search(Xfloat_N12,Pfloat_N12,id_use)
        if(sub_id_use):
            #返回部分模糊度固定信息
            N12_SD_FIX_info={"ratios":[t_ratios[0],t_ratios[1]]}
            N12_SD_FIX_info['Ref_sat']=el_max_prns
            N12_SD_FIX_info['N12_SD_sat']=[]
            N12_SD_FIX_info['N12_SD_value']=[]
            for id in sub_id_use:
                #L1频率上的星间单差模糊度
                if (id<len(rnx_obs)):
                    N12_SD_FIX_info['N12_SD_sat'].append([rnx_obs[id]['PRN'],'N1'])
                #L2频率上的星间单差模糊度
                else:
                    N12_SD_FIX_info['N12_SD_sat'].append([rnx_obs[id-len(rnx_obs)]['PRN'],'N2'])
                N12_SD_FIX_info['N12_SD_value'].append(N12_FIX[id_use.index(id)])
            return N12_SD_FIX_info
        return False

def PPP_AR_FIX_HOLD_M(t_Xk,t_Pk,rnx_obs,Fix_info,X,freqs):
    #利用虚拟观测更新固定解
    #计算单系统卫星数量
    sys_sat_num=round((X.shape[0]-5)/3)
    prns=[t['PRN'] for t in rnx_obs]#获取本历元观测卫星的PRN顺序
    #添加固定解虚拟约束
    fix_value_count=0
    H=np.zeros( (len(Fix_info['N12_SD_sat']),len(t_Xk)) )   #虚拟设计矩阵维数: (固定模糊度行, 本历元状态量列)
    v=np.zeros( (len(Fix_info['N12_SD_sat']),1) )           #虚拟观测值维数: (固定模糊度行,1)
    R=np.zeros( (len(Fix_info['N12_SD_sat']),len(Fix_info['N12_SD_sat'])) )
    for fix in Fix_info['N12_SD_sat']:
        fix_prn=fix[0]
        fix_freq=int(fix[1][1:])                                      #计算模糊度频率所属
        ref_prn=Fix_info['Ref_sat'][['G','C','E'].index(fix_prn[0])]     #查找参考星
        sat_id=prns.index(fix_prn)                                      #在本历元观测序列中查找目标卫星位置
        ref_sat_id=prns.index(ref_prn)                                  #在本历元观测序列中查找参考卫星位置

        freq=freqs[['G','C','E'].index(fix_prn[0])][fix_freq-1]#频率分发
        
        H[fix_value_count][5+fix_freq*len(rnx_obs)+sat_id]=1.0
        H[fix_value_count][5+fix_freq*len(rnx_obs)+ref_sat_id]=-1.0
        
        #计算模糊度单差的残差, 对残差向量添加一行
        sys_shift=[0,3*32,3*(32+65)][['G','C','E'].index(fix_prn[0])]
        sys_snum=[32,65,37][['G','C','E'].index(fix_prn[0])]
        ta=X[5+sys_shift+fix_freq*sys_snum+int(fix_prn[1:])-1][0]
        tb=X[5+sys_shift+fix_freq*sys_snum+int(ref_prn[1:])-1][0]
        float_v=ta-tb
        fix_v=round(Fix_info['N12_SD_value'][fix_value_count])*clight/freq-(float_v)
        v[fix_value_count][0]=fix_v #固定解约束
        R[fix_value_count][fix_value_count]=0.001 #虚拟方差
        fix_value_count+=1      #虚拟观测序号+1
    #1.状态一步预测
    X_up=t_Xk
    #2.状态一步预测误差
    Pk_1_k=t_Pk
    #3.滤波增益计算
    Kk=(Pk_1_k.dot(H.T)).dot(inv((H.dot(Pk_1_k)).dot(H.T)+R))
    #Kk=(Pk_1_k.dot(H.T)).dot(numba_inv((H.dot(Pk_1_k)).dot(H.T)+R))
    #滤波结果
    Xk_dot=X_up+Kk.dot(v)
    #滤波方差更新
    t_Pk=((np.eye(t_Xk.shape[0],dtype=np.float64)-Kk.dot(H))).dot(Pk_1_k)  
    t_Pk=t_Pk.dot((np.eye(t_Xk.shape[0],dtype=np.float64)-Kk.dot(H)).T)+Kk.dot(R).dot(Kk.T)
    t_Xk=Xk_dot

    return Xk_dot,t_Pk  #返回滤波更新后的值


def add_PPP_RTK_corr(H,R,v,corr_v,corr_sstd,ref_ids,rnx_obs):
    #函数: PPP-RTK约束函数
    #输入: 非约束观测模型HRv, 约束向量corr_v, 约束向量的方差corr_sstd
    #输出: 有约束观测模型H_corr, R_corr, v_corr
    H_corr=H.copy()
    R_corr=R.copy()
    v_corr=v.copy()
    prns=[t['PRN'] for t in rnx_obs]
    for i in range(len(corr_v)):        #顺序为卫星列表顺序

        if(corr_v[i]==0.0):
            continue                    #该状态量无改正数
        H_corr=np.append(H_corr,np.zeros((1,H_corr.shape[1])),axis=0)#设计矩阵添加一行
        H_corr[-1][i]=1.0                                            #约束对应状态的系数置1(基准站位置约束)
        
        if(i>=5):                                                    #电离层约束
            prn=prns[i-5]                     #卫星PRN码
            sys_id=['G','C','E'].index(prn[0])
            H_corr[-1][5+ref_ids[sys_id]]=-1.0                       #参考星
        v_corr=np.append(v_corr,np.array([[corr_v[i]]]),axis=0)      #残差向量添加一行
        
        #观测噪声阵向右下添加一块
        R_corr=np.append(R_corr,np.zeros((1,R_corr.shape[1])),axis=0)   #先添加一行
        R_corr=np.append(R_corr,np.zeros((R_corr.shape[0],1)),axis=1)   #再添加一列
        R_corr[-1][-1]=corr_sstd[i]                                #虚拟观测值赋权重
    
    #返回添加PPP-RTK改正数约束后的观测模型
    return H_corr,R_corr,v_corr

#大气约束
def rtkinfo2SIONTRO(rtk_info,freqs,Qi_init=2.0):
    SION,TRO={},{}
    for prn in rtk_info.keys():
        #1. 剔除低高度角改正数
        try:
            el=rtk_info[prn]['azel'][1]
            if(el<10.0):
                continue
        except:
            pass
        #2. 分发频率
        f1=freqs[['G','C','E'].index(prn[0])][0]
        #3. 构建电离层延迟改正数群组
        SION[prn]=[rtk_info[prn]['STEC']*40.28/(f1/1e8)/(f1/1e8),Qi_init**2]
        try:
            SION[prn][1]=(rtk_info[prn]['std_STEC']+Qi_init**2)*(40.28/(f1/1e8)/(f1/1e8))**2
        except:
            pass
    #3. 分发基准站位置 (用于定权)
    base_pos=[0,0,0]
    if(rtk_info!={}):
        base_x=rtk_info[list(rtk_info.keys())[0]]['sta_x']
        base_y=rtk_info[list(rtk_info.keys())[0]]['sta_y']
        base_z=rtk_info[list(rtk_info.keys())[0]]['sta_z']
        base_pos=[base_x,base_y,base_z]
        try:
            TRO['ZTD']=rtk_info[list(rtk_info.keys())[0]]['ztd_w']+rtk_info[list(rtk_info.keys())[0]]['ztd_h']    #含模型值的ZTD必须传递原始值
            TRO['ZTD-Q']=rtk_info[list(rtk_info.keys())[0]]['std_ztd_w']    #利用估计量的代替模型值
        except:
            TRO['ZTD']=get_Trop_delay_dry(base_pos)+0.15
            TRO['ZTD-W-Q']=1.0
    
    return SION,TRO,base_pos

def get_IPP_rad(rr,rs):
    #B2b信号推荐参数设置
    H_ion=350000            #电离层薄层高度
    Re=6378000              #地球平均半径
    lng_M=-72.58/180*pi     #地磁北极的地理经度
    lat_M=80.27/180*pi      #地磁北极的地理纬度
    
    #站星射线与高度角计算
    #Calculation of star ray and height angle at station 
    az,el=getazel(rs,rr)
    #测站经纬度
    rrb,rrl,rrh=xyz2blh(rr[0],rr[1],rr[2])
    rrb=rrb/180*pi
    rrl=rrl/180*pi
    #电离层穿刺点地心张角计算
    PHI=pi/2-el-asin(Re/(Re+H_ion)*cos(el))
    #电离层穿刺点在地球表面投影的地理经纬度
    lat_g=asin( sin(rrb)*cos(PHI)+cos(rrb)*sin(PHI)*cos(az) )
    lng_g=rrl+atan2(sin(PHI)*sin(az)*cos(rrb),cos(PHI)-sin(rrb)*sin(lat_g))
    #电离层延迟在地球表面投影的地磁经纬度
    lat_m=asin(sin(lat_M)*sin(lat_g) + cos(lat_M)*cos(lat_g)*cos(lng_g-lng_M) )
    lng_m=atan2(cos(lat_g)*sin(lng_g-lng_M)*cos(lat_M) , sin(lat_M)*sin(lat_m)-sin(lat_g) )
    return lat_g,lng_g

def caculate_PPP_RTK_corr_M(rnx_obs, X, pos=[], TRO={}, SION={}, peph_sat_pos={}, base_pos=None, rove_pos=[], Qi_scale=10e3, Qi_ele_threshold=10,Qt_scale=10e6):
    # 函数: 计算PPP_RTK约束向量与方差向量
    # 输入: 有效观测列表rnx_obs, 位置约束(初始化为空), 大气约束(初始化为空字典), 电离层约束(初始化为空字典),  
    # 输出: PPP_RTK约束向量corr_v, 方差向量corr_sstd
    corr_v=[]
    corr_sstd=[]
    sys_sat_num=round((len(X)-5)/3)
    #位置约束
    if(len(pos)!=3):
        corr_v.append(0.0)
        corr_v.append(0.0)
        corr_v.append(0.0)
        corr_sstd.append(0.001)
        corr_sstd.append(0.001)
        corr_sstd.append(0.001)
    else:
        corr_v.append(pos[0]-X[0][0])
        corr_v.append(pos[1]-X[1][0])
        corr_v.append(pos[2]-X[2][0])
        corr_sstd.append(0.001)
        corr_sstd.append(0.001)
        corr_sstd.append(0.001)
    
    #钟差约束预留位置
    corr_v.append(0.0)
    corr_sstd.append(0.0)

    #对流层约束(当前仅支持ZWD加入约束)
    if(TRO!={} and Qt_scale!=-1):
        baseline=sqrt( (base_pos[0]-X[0][0])**2+(base_pos[1]-X[1][0])**2+(base_pos[2]-X[2][0])**2 )
        corr_v.append((TRO['ZTD']-get_Trop_delay_dry(rove_pos))-X[4][0])
        #corr_v.append(0.0)
        corr_sstd.append(TRO['ZTD-Q']+(baseline/Qt_scale)**2)#以10cm为方差
        #corr_sstd.append(1.0*1.0)
    else:
        corr_v.append(0.0)
        corr_sstd.append(0.0)
    
    #电离层约束(星间单差模式)
    #首先选择参考星
    ref_prns=['X00','X00','X00']
    el_maxs=[0,0,0]
    ref_ids=[999,999,999]
    count_id=0
    for sat in rnx_obs:
        prn=sat["PRN"]
        _,el=getazel(peph_sat_pos[prn],[X[0][0],X[1][0],X[2][0]])   #获取高度角
        if(prn not in SION.keys()):                            #无SSR的卫星跳过
            count_id+=1
            continue
        if(el>el_maxs[0] and prn[0]=='G'):
            el_maxs[0]=el
            ref_prns[0],ref_ids[0]=prn,count_id
        if(el>el_maxs[1] and prn[0]=='C'):
            el_maxs[1]=el
            ref_prns[1],ref_ids[1]=prn,count_id
        if(el>el_maxs[2] and prn[0]=='E'):
            el_maxs[2]=el
            ref_prns[2],ref_ids[2]=prn,count_id
        count_id+=1
    
    for sat in rnx_obs:
        prn=sat['PRN']
        sys_id=['G','C','E'].index(prn[0])
        sys_shift=[0,3*32,3*(32+65)]
        sat_count=int(prn[1:])-1
        ref_count=int(ref_prns[sys_id][1:])-1
        _,el=getazel(peph_sat_pos[prn],[X[0][0],X[1][0],X[2][0]])
        try:
            sion,sion_r=SION[prn][0],SION[ref_prns[sys_id]][0]              #基准站站解ION(单位为m)
            sion_q,sion_r_q=SION[prn][1],SION[ref_prns[sys_id]][1]          #基准站电离层质量因子(单位为m)
            ion,ion_r=X[5+sys_shift[sys_id]+sat_count][0],X[5+sys_shift[sys_id]+ref_count][0]
            #电离层状态方差信息
            Qi0=sion_q#+sion_r_q                                #基准站质量因子
            lat_b,lng_b=get_IPP_rad(base_pos,peph_sat_pos[prn])
            lat_r,lng_r=get_IPP_rad(rove_pos,peph_sat_pos[prn])
            Qid=(float(Qi_scale)**2)*((lat_b-lat_r)**2+(lng_b-lng_r)**2)        #基线质量衰减因子
            Qi1=Qid/sin(el)/sin(el)                             #高度角加权因子
            #所有约束信息校验通过, 添加约束和约束方差
            if(Qi_scale!=-1 and el/pi*180.0>=Qi_ele_threshold):
                corr_v.append((sion-sion_r)-(ion-ion_r))                        #虚拟观测约束添加
                corr_sstd.append(Qi0+Qi1)                           #电离层SSR约束方差
            else:
                corr_sstd.append(0.0)
                corr_v.append(0.0)
        except:
            corr_v.append(0.0)
            corr_sstd.append(0.0)
    return corr_v,corr_sstd,ref_ids


def KF_UCPPP_M(X,X_time,Pk,Qk,ion_param,peph_sat_pos,rnx_obs,ex_threshold_v,ex_threshold_sigma,rt_unix,phase_bias,freqs,AMB_FIX=1,mode="None",RTK_Info={},AMB_FIX_Info={},Out_log_fix=[]):
    #扩展卡尔曼滤波
    for KF_times in range(8):
        #相位改正值拷贝
        t_phase_bias=phase_bias.copy()
        
        #观测模型(两次构建, 验前粗差剔除)
        #print(rnx_obs)
        X,X_time,H,R,_,v,rnx_obs=createKF_HRZ_M(rnx_obs,rt_unix,X,X_time,Pk,Qk,ion_param,t_phase_bias,peph_sat_pos,freqs=freqs,exthreshold_v_sigma=ex_threshold_sigma,post=False,ex_threshold_v=ex_threshold_v)
        if(not len(rnx_obs)):
            #无先验通过数据
            #全部状态重置
            X[0]=100.0
            X[1]=100.0
            X[2]=100.0
            # for i in range(len(X)):
            #     X_time[i]=0.0
            #     break
        X,X_time,H,R,_,v,rnx_obs=createKF_HRZ_M(rnx_obs,rt_unix,X,X_time,Pk,Qk,ion_param,t_phase_bias,peph_sat_pos,freqs=freqs,exthreshold_v_sigma=ex_threshold_sigma,post=False,ex_threshold_v=ex_threshold_v)

        #基准站约束
        if(mode=='Base'):
            STA_P=RTK_Info['STA_P']
            STA_Q=RTK_Info['STA_Q']
            corr_v=[STA_P[0]-X[0][0], STA_P[1]-X[1][0], STA_P[2]-X[2][0]]
            corr_sstd=[STA_Q[0],STA_Q[1],STA_Q[2]]
            ref_id=0
            H,R,v=add_PPP_RTK_corr(H,R,v,corr_v,corr_sstd,[],rnx_obs)
        #流动站约束
        if(mode=='Rove'):
            _,GPSsec=time2gpst(rt_unix)
            t_interval=RTK_Info['t_interval']
            GPSsec=round(GPSsec/t_interval)*t_interval      #SSR时间戳
            try:
                rtk_info_id=RTK_Info['rtk_corr_info_time'].index(GPSsec)
                rtk_info=RTK_Info['rtk_info'][rtk_info_id]#查找目标时间对应的SSR组
            except:
                rtk_info={}
            SION,TRO,base_pos=rtkinfo2SIONTRO(rtk_info,freqs,Qi_init=RTK_Info['Qi_init'])
            rove_pos=[X[0][0],X[1][0],X[2][0]]
            corr_v,corr_sstd,ref_ids=caculate_PPP_RTK_corr_M(rnx_obs,X,TRO=TRO,SION=SION,peph_sat_pos=peph_sat_pos,base_pos=base_pos,rove_pos=rove_pos,Qi_scale=RTK_Info['Qi_scale'],Qi_ele_threshold=RTK_Info['Qi_ele_threshold'],Qt_scale=RTK_Info['Qt_scale'])
            H,R,v=add_PPP_RTK_corr(H,R,v,corr_v,corr_sstd,ref_ids=ref_ids,rnx_obs=rnx_obs)
            

        #系统模型(根据先验抗差得到的数据)
        t_Xk,t_Pk,t_Qk=createKF_XkPkQk_M(rnx_obs,X,Pk,Qk)

        #抗差滤波准备
        #R=IGGIII(v,R)
        #扩展卡尔曼滤波
        #1.状态一步预测
        PHIk_1_k=np.eye(t_Xk.shape[0],dtype=np.float64)
        X_up=PHIk_1_k.dot(t_Xk)
        #2.状态一步预测误差
        Pk_1_k=(PHIk_1_k.dot(t_Pk)).dot(PHIk_1_k.T)+t_Qk
        #3.滤波增益计算
        #Kk=(Pk_1_k.dot(H.T)).dot(inv((H.dot(Pk_1_k)).dot(H.T)+R))
        Kk=(Pk_1_k.dot(H.T)).dot(numba_inv((H.dot(Pk_1_k)).dot(H.T)+R))
        #滤波结果
        Xk_dot=X_up+Kk.dot(v)
        #滤波方差更新
        t_Pk=((np.eye(t_Xk.shape[0],dtype=np.float64)-Kk.dot(H))).dot(Pk_1_k)  
        t_Pk=t_Pk.dot((np.eye(t_Xk.shape[0],dtype=np.float64)-Kk.dot(H)).T)+Kk.dot(R).dot(Kk.T)
        #滤波暂态更新
        t_Xk_f,t_Pk_f,t_Qk_f,t_X_time=upstateKF_XkPkQk_M(rnx_obs,rt_unix,Xk_dot,t_Pk,t_Qk,X,Pk,Qk,X_time)
        
        #验后方差
        info='KF fixed'
        info,rnx_obs,tt_phase_bias,v=createKF_HRZ_M(rnx_obs,rt_unix,t_Xk_f,t_X_time,t_Pk_f,t_Qk_f,ion_param,t_phase_bias,peph_sat_pos,freqs=freqs,exthreshold_v_sigma=ex_threshold_sigma,post=True)
        #_,info=get_post_v(rnx_obs,rt_unix,Xk_dot,ion_param,phase_bias)
        
        if(info=='KF fixed'):    
            #验后校验通过
            X,Pk,Qk,X_time=upstateKF_XkPkQk_M(rnx_obs,rt_unix,Xk_dot,t_Pk,t_Qk,X,Pk,Qk,X_time)
            phase_bias=tt_phase_bias.copy()
            #不固定模糊度
            if(not AMB_FIX):
                break
            #固定模糊度
            try:
                Fix_info=PPP_AR_M(rnx_obs,peph_sat_pos,Xk_dot,t_Pk,freqs,ratio_threshold=AMB_FIX_Info['ratio_threshold'],
                                  amb_float_threshold=AMB_FIX_Info['amb_float_threshold'],
                                  P_float_threshold=AMB_FIX_Info['P_float_threshold'],
                                  min_AR_num=AMB_FIX_Info['min_AR_num'])
                if(Fix_info!=False):
                    #滤波法更新模糊度
                    Xk_dot,t_Pk=PPP_AR_FIX_HOLD_M(Xk_dot,t_Pk,rnx_obs,Fix_info,X,freqs)
                    #验后校验通过
                    t_X,t_P,Qk,X_time=upstateKF_XkPkQk_M(rnx_obs,rt_unix,Xk_dot,t_Pk,t_Qk,X,Pk,Qk,X_time)
                    #print(xyz2neu(X_SD_fix,[ -2267750.307108253, 5009154.576877178, 3221294.4087844505]))
                    Out_log_fix.append(log2out_M(rt_unix,v,rnx_obs,t_X,t_X_time,t_P,peph_sat_pos,freqs,Fix_info['ratios'][0][0][0]))
                    pass
                else:
                #固定失败, 保存浮点解
                     Out_log_fix.append(log2out_M(rt_unix,v,rnx_obs,X,X_time,Pk,peph_sat_pos,freqs))
            except:
                Out_log_fix.append(log2out_M(rt_unix,v,rnx_obs,X,X_time,Pk,peph_sat_pos,freqs))
                pass
            break
    if(info!='KF fixed'):
        print("Warning: KF itertion overflow")

    return X,Pk,Qk,X_time,v,phase_bias,rnx_obs


def find_OSB_index(filename):
    targets=["BIAS","SVN","PRN","STATION","OBS1","OBS2","BIAS_START","BIAS_END","UNIT","ESTIMATED_VALUE","STD_DEV"]
    target_ids=[-1 for t in range(len(targets))]
    target_ide=[-1 for t in range(len(targets))]
    with open(filename,'r',encoding='gbk') as f:
        lines=f.readlines()
        bias_in=0
        for line in lines:
            if ("+BIAS/SOLUTION" in line):
                bias_in=1
            if(bias_in and line[0]=='*'):
                ls=line.split()
                for t in range(len(targets)):
                    target=targets[t]
                    #循环字符查找
                    for i in range(len(ls)):
                        #找到目标索引
                        if(target in ls[i]):
                            for j in range(len(line)-len(ls[i])):
                                #字符切片匹配成功
                                if(line[j:j+len(ls[i])]==ls[i]):
                                    target_ids[t]=j
                                    target_ide[t]=j+len(ls[i])
                                    break
                            break
            if("-BIAS/SOLUTION" in line):
                bias_in=0
                break
    return targets,target_ids,target_ide


def OSBcorr2obsmat(filename,obs_mat,obs_type,f1,f2):
    
    #首先确定索引
    targets,target_ids,target_ide=find_OSB_index(filename)
    id_osb=targets.index('BIAS')
    id_obs=targets.index('OBS1')
    id_prn=targets.index('PRN')
    id_value=targets.index('ESTIMATED_VALUE')
    
    with open(filename,'r',encoding='gbk') as f:
        data={}    
        lines=f.readlines()
        bias_in=0
        for line in lines:
            #数据段判断
            if ("+BIAS/SOLUTION" in line):
                bias_in=1
            ls=line.split()
            if(not bias_in):
                continue
            #记录OSB字典
            if('OSB' in line[target_ids[id_osb]:target_ide[id_osb]]):#找到OSB行
                prn_c=line[target_ids[id_prn]:target_ide[id_prn]]
                if(prn_c not in data.keys()):
                    data[prn_c]={}
                obs_sign=line[target_ids[id_obs]:target_ide[id_obs]].replace(" ","")
                osb_value=line[target_ids[id_value]:target_ide[id_value]]
                data[prn_c][obs_sign]=float(osb_value)*clight*1e-9
            
            if ("-BIAS/SOLUTION" in line):
                bias_in=0
                break

    for i in range(len(obs_mat)):
        for sat in range(len(obs_mat[i][1])):
            prn=obs_mat[i][1][sat]['PRN']
            if(obs_mat[i][1][sat]['OBS'][0]!=0.0):
                try:
                    obs_mat[i][1][sat]['OBS'][0]-=data[prn][obs_type[0]]
                except:
                    pass
            if(obs_mat[i][1][sat]['OBS'][1]!=0.0):
                try:
                        obs_mat[i][1][sat]['OBS'][1]-=data[prn][obs_type[1]]*f1/clight
                except:
                    pass
            if(obs_mat[i][1][sat]['OBS'][5]!=0.0):
                try:
                    obs_mat[i][1][sat]['OBS'][5]-=data[prn][obs_type[4]]
                except:
                    pass
            if(obs_mat[i][1][sat]['OBS'][6]!=0.0):
                try:
                    obs_mat[i][1][sat]['OBS'][6]-=data[prn][obs_type[5]]*f2/clight
                except:
                    pass
    return obs_mat

In [28]:
def PPP_YAML_GCE(PPP_cfg):
    #首先可视化配置
    print("Easy4PPP Configurations:")
    for key in PPP_cfg.keys():
        print(key,PPP_cfg[key])
    
    #多系统双频非差非组合PPP, 解算文件最小配置(观测值/观测值类型/精密星历文件/精密钟差文件/天线文件/结果输出路径)
    obs_file=PPP_cfg['obs_file']

    sys_indexs=PPP_cfg['sys_indexs']
    sys_select_num=len(sys_indexs)
    sys_select_ids=sys_indexs.copy()                               #用户选择的原始系统标识
    sys_indexs=['G','C','E']                                       #重置系统标识

    obs_type=PPP_cfg['obs_type']                                   #混合观测值类型, 以RINEX协议为准
    freqs=PPP_cfg['freqs']                                         #各频点观测值中央频率
    
    #系统与信号标识符校验
    try:
        if(len(sys_indexs)==len(obs_type)):
            print("Systems set as: ",sys_indexs)
        else:
            print("Systems set error for: ")
            print("sys_indexs: ", sys_indexs)
            print("obs_type: ",   obs_type)
    except:
        ValueError("Systems not set correctly")

    #重整观测值列表和频率数组
    if(sys_select_num):
        try:
            freqs_G=freqs[sys_select_ids.index("G")]
            obs_type_G=obs_type[sys_select_ids.index("G")]
        except:
            freqs_G=[1575.42E+6, 1227.60E+6]#默认L1/L2
            obs_type_G=['C1C','L1C','D1C','S1C','C2W','L2W','D2W','S2W']
        try:
            freqs_C=freqs[sys_select_ids.index("C")]
            obs_type_C=obs_type[sys_select_ids.index("C")]
        except:
            freqs_C=[1561.098E+6,1268.52E+6]#默认B1/B3
            obs_type_C=['C2I','L2I','D2I','S2I','C6I','L6I','D6I','S6I']
        try:
            freqs_E=freqs[sys_select_ids.index("E")]
            obs_type_E=obs_type[sys_select_ids.index("E")]
        except:
            freqs_E=[1575.42E+6, 1176.45E+6]#默认E1/E5
            obs_type_E=['C1X','L1X','D1X','S1X','C5X','L5X','D5X','S5X']
        freqs=[freqs_G,freqs_C,freqs_E]
        obs_type=[obs_type_G,obs_type_C,obs_type_E]
    
    SP3_file=PPP_cfg['SP3_file']                                #精密星历文件路径
    CLK_file=PPP_cfg['CLK_file']                                #精密钟差文件路径
    ATX_file=PPP_cfg['ATX_file']                                #天线文件, 支持格式转换后的npy文件和IGS天线文件

    out_path=PPP_cfg['out_path']                                        #导航结果输出文件路径
    print("Navigation Results saved in path: ",out_path)
    ion_param=PPP_cfg['ion_param']                                                 #自定义Klobuchar电离层参数
    
    if(len(ion_param)):
        print("ion_params set as: ",ion_param)
    else:
        print("No ion_param, read from file. ")
    
    #可选配置(广播星历文件/DCB改正与产品选项/)
    BRDC_file=PPP_cfg['BRDC_file']                             #广播星历文件, 支持BRDC/RINEX3混合星历
    print("Broadcast eph file: ",BRDC_file)
    dcb_correction=PPP_cfg['dcb_correction']                   #DCB修正选项
    dcb_products=PPP_cfg['dcb_products']                       #DCB产品来源, 支持CODE月解文件/CAS日解文件
    dcb_file_0=PPP_cfg['dcb_file_0']                           #频间偏差文件, 支持预先转换后的.npy格式
    dcb_file_1=PPP_cfg['dcb_file_1']
    dcb_file_2=PPP_cfg['dcb_file_2']

    obs_start=PPP_cfg['obs_start']                             #解算初始时刻索引
    obs_epoch=PPP_cfg['obs_epoch']                             #解算总历元数量
    out_age=PPP_cfg['out_age']                                 #最大容忍失锁阈值时间(单位: s, 用于电离层、模糊度状态重置)
    dy_mode=PPP_cfg['dy_mode']                                 #PPP动态模式配置, 支持static, dynamic
    el_threthod=PPP_cfg['el_threthod']                         #设置截止高度角
    ex_threshold_v=PPP_cfg['ex_threshold_v']                   #设置先验残差阈值
    ex_threshold_v_sigma=PPP_cfg['ex_threshold_v_sigma']       #设置后验残差阈值
    Mw_threshold=PPP_cfg['Mw_threshold']                       #设置Mw组合周跳检验阈值
    GF_threshold=PPP_cfg['GF_threshold']                       #设置GF组合周跳检验阈值
    sat_out=PPP_cfg['sat_out']

    #模糊度固定
    try:
        AMB_FIX=PPP_cfg['AMB_FIX']
        OSB_path=PPP_cfg['OSB_path']
        ratio_threshold=PPP_cfg['ratio_threshold']
        amb_float_threshold=PPP_cfg['amb_float_threshold']
        P_float_threshold=PPP_cfg['P_float_threshold']
        min_AR_num=PPP_cfg['min_AR_num']
    except:
        AMB_FIX=0
        OSB_path=""
        ratio_threshold=2.0
        amb_float_threshold=1.0
        P_float_threshold=2.25
        min_AR_num=3
    AMB_FIX_Info={'ratio_threshold':ratio_threshold,'amb_float_threshold':amb_float_threshold,
                      'P_float_threshold':P_float_threshold,'min_AR_num':min_AR_num}    
    
    #OSB校正
    try:
        OSB_YES=PPP_cfg['OSB_YES']
        PCO_YES=PPP_cfg['PCO_YES']
    except:
        OSB_YES=0
        PCO_YES=1   #默认改正PCO

    #基准站信息
    try:
        sta_mode=PPP_cfg['sta_mode']
        STA_P=PPP_cfg['STA_P']
        STA_Q=PPP_cfg['STA_Q']
    except:
        sta_mode='None'
        STA_P=None
        STA_Q=None

    #整理PPP-RTK信息
    RTK_Info={}
    try:
        RTK_Info['reinitial_sec']=PPP_cfg['reinitial_sec']  #记录重收敛信息
    except:
        RTK_Info['reinitial_sec']=0                         #没有重收敛信息
    
    if(sta_mode=='Base'):
        RTK_Info['STA_P']=STA_P
        RTK_Info['STA_Q']=STA_Q
    if(sta_mode=='Rove'):
        RTK_Info['t_interval']=PPP_cfg['t_interval']
        RTK_Info['rtk_info']=np.load(PPP_cfg["rtk_info_mat"],allow_pickle=True)
        
        rtk_corr_info_time=[]
        for log in RTK_Info['rtk_info']:
            try:
                rtk_corr_info_time.append(log[list(log.keys())[0]]['GPSsec'])
            except:
                rtk_corr_info_time.append(9999999)
        RTK_Info['rtk_corr_info_time']=rtk_corr_info_time
        RTK_Info['Qi_init']=PPP_cfg['Qi_init']
        RTK_Info['Qi_scale']=PPP_cfg['Qi_scale']
        RTK_Info['Qi_ele_threshold']=PPP_cfg['Qi_ele_threshold']
        RTK_Info['Qt_scale']=PPP_cfg['Qt_scale']
    
    #处理非GCE系统情况:
    if("G" not in sys_select_ids):
        for i in range(1,33):
            sat_out.append("G{:02d}".format(i))
    if("C" not in sys_select_ids):
        for i in range(1,66):
            sat_out.append("C{:02d}".format(i))
    if("E" not in sys_select_ids):
        for i in range(1,37):
            sat_out.append("E{:02d}".format(i))
    
    #
    #多系统观测值分别读取
    STA_name=obs_file.split('/')[-1][:4].upper()
    print("The name of station (RINEX observation format): ",STA_name)
    dcb_file_0_=""
    if "G" in sys_indexs:
        #CAS DCB产品数据读取
        if(dcb_correction==1 and dcb_products=='CAS'):
            dcb_file_0_,_=CAS_DCB_SR(dcb_file_0,obs_type[0][0],obs_type[0][4],STA_name)
            dcb_file_1=''       #CAS产品同时包含码间和频间偏差
            dcb_file_2=''
        obs_mat_GPS=RINEX3_to_obsmat(obs_file,obs_type[0],sys="G",dcb_correction=dcb_correction,dcb_file_0=dcb_file_0_,dcb_file_1=dcb_file_1,dcb_file_2=dcb_file_2)
        obs_mat_GPS=reconstruct_obs_mat(obs_mat_GPS)
        if(AMB_FIX or OSB_YES):
            obs_mat_GPS=OSBcorr2obsmat(OSB_path,obs_mat_GPS,obs_type[0],freqs_G[0],freqs_G[1])
        #删除CAS-DCB产品辅助文件
        if(dcb_file_0_!=""):
            os.unlink(dcb_file_0_)
    if "C" in sys_indexs:
        if(dcb_correction==1 and dcb_products=='CAS'):
            dcb_file_0_,_=CAS_DCB_SR(dcb_file_0,obs_type[1][0],obs_type[1][4],STA_name)
            dcb_file_1=''       #CAS产品同时包含码间和频间偏差
            dcb_file_2=''   
        obs_mat_BDS=RINEX3_to_obsmat(obs_file,obs_type[1],sys="C",dcb_correction=dcb_correction,dcb_file_0=dcb_file_0_,dcb_file_1=dcb_file_1,dcb_file_2=dcb_file_2)
        obs_mat_BDS=reconstruct_obs_mat(obs_mat_BDS)
        if(AMB_FIX or OSB_YES):
            obs_mat_BDS=OSBcorr2obsmat(OSB_path,obs_mat_BDS,obs_type[1],freqs_C[0],freqs_C[1])
        if(dcb_file_0_!=""):
            os.unlink(dcb_file_0_)
    if "E" in sys_indexs:
        if(dcb_correction==1 and dcb_products=='CAS'):
            dcb_file_0_,_=CAS_DCB_SR(dcb_file_0,obs_type[2][0],obs_type[2][4],STA_name)
            dcb_file_1=''       #CAS产品同时包含码间和频间偏差
            dcb_file_2=''
        obs_mat_GAL=RINEX3_to_obsmat(obs_file,obs_type[2],sys="E",dcb_correction=dcb_correction,dcb_file_0=dcb_file_0_,dcb_file_1=dcb_file_1,dcb_file_2=dcb_file_2)
        obs_mat_GAL=reconstruct_obs_mat(obs_mat_GAL)
        if(AMB_FIX or OSB_YES):
            obs_mat_GAL=OSBcorr2obsmat(OSB_path,obs_mat_GAL,obs_type[2],freqs_E[0],freqs_E[1])
        if(dcb_file_0_!=""):
            os.unlink(dcb_file_0_)
    #读取观测文件
    if (not check_obs_mats([obs_mat_GPS,obs_mat_BDS,obs_mat_GAL])):
        ValueError()

    if(not obs_epoch):
        obs_epoch=len(obs_mat_GPS)
        print("End index not set, solve all the observations. Total: {}".format(obs_epoch))
    
    #读取精密轨道和钟差文件
    IGS=getsp3(SP3_file)
    clk=getclk(CLK_file)
    
    #读取天线文件
    try:
        #npy格式
        sat_pcos=np.load(ATX_file,allow_pickle=True)
        sat_pcos=eval(str(sat_pcos))
    except:
        #ATX格式
        sat_pcos=RINEX3_to_ATX(ATX_file)
    
    if(PCO_YES==0):
        sat_pcos={}
    
    #读取广播星历电离层参数
    if(not len(ion_param)):
        ion_param=RINEX2ion_params(BRDC_file)

    
    #根据配置设置卫星数量
    sat_num=0
    sat_num_G=0
    sat_num_C=0
    sat_num_E=0
    if('G' in sys_indexs):
        sat_num_G=32
    if('C' in sys_indexs):
        sat_num_C=65
    if('E' in sys_indexs):
        sat_num_E=37
    sat_num=sat_num_G+sat_num_C+sat_num_E
    print("Total satellite number of selected systems: ", sat_num," GPS: ",sat_num_G," BDS: ",sat_num_C," GAL",sat_num_E)
    
    #排除卫星列表
    print("Satellites outside for user config",sat_out)
    
    # 分别初始化各系统PPP子滤波状态与协方差
    X_G,Pk_G,Qk_G,GF_sign_G,Mw_sign_G,slip_sign_G,dN_sign_G,X_time_G,phase_bias_G=init_UCPPP(obs_mat_GPS,obs_start,IGS,clk,sat_out,ion_param,sat_pcos,sys_sat_num=sat_num_G,f1=freqs[0][0],f2=freqs[0][1])
    X_C,Pk_C,Qk_C,GF_sign_C,Mw_sign_C,slip_sign_C,dN_sign_C,X_time_C,phase_bias_C=init_UCPPP(obs_mat_BDS,obs_start,IGS,clk,sat_out,ion_param,sat_pcos,sys_sat_num=sat_num_C,f1=freqs[1][0],f2=freqs[1][1])
    X_E,Pk_E,Qk_E,GF_sign_E,Mw_sign_E,slip_sign_E,dN_sign_E,X_time_E,phase_bias_E=init_UCPPP(obs_mat_GAL,obs_start,IGS,clk,sat_out,ion_param,sat_pcos,sys_sat_num=sat_num_E,f1=freqs[2][0],f2=freqs[2][1])
    
    #多系统非差PPP滤波状态初始化
    X,Pk,Qk,GF_sign,Mw_sign,slip_sign,dN_sign,X_time,phase_bias=init_UCPPP_M(X_G,X_C,X_E,
                 Pk_G,Pk_C,Pk_E,
                 Qk_G,Qk_C,Qk_E,
                 GF_sign_G,GF_sign_C,GF_sign_E,
                 Mw_sign_G,Mw_sign_C,Mw_sign_E,
                 slip_sign_G,slip_sign_C,slip_sign_E,
                 dN_sign_G,dN_sign_C,dN_sign_E,
                 X_time_G,X_time_C,X_time_E,
                 phase_bias_G,phase_bias_C,phase_bias_E)

    Out_log,Out_log_fix=UCPPP_M([obs_mat_GPS,obs_mat_BDS,obs_mat_GAL],obs_start,obs_epoch,IGS,clk,
          sat_out,ion_param,sat_pcos,el_threthod,ex_threshold_v,ex_threshold_v_sigma,Mw_threshold,GF_threshold,dy_mode,
          X,Pk,Qk,phase_bias,X_time,GF_sign,Mw_sign,slip_sign,dN_sign,sat_num,out_age,freqs,AMB_FIX,sta_mode,RTK_Info=RTK_Info,AMB_FIX_Info=AMB_FIX_Info)

    #结果以numpy数组格式保存在指定输出目录下, 若输出目录为空, 则存于nav_result
    try:
        np.save(out_path+'/{}.out'.format(os.path.basename(obs_file)),Out_log,allow_pickle=True)
        print("Navigation results saved at ",out_path+'/{}.out'.format(os.path.basename(obs_file)))
        if(AMB_FIX):
            np.save(out_path+'/{}.fixout'.format(os.path.basename(obs_file)),Out_log_fix,allow_pickle=True)
            print("Navigation results (AR) saved at ",out_path+'/{}.fixout'.format(os.path.basename(obs_file)))
    except:
        np.save('nav_result/{}.out'.format(os.path.basename(obs_file)),Out_log,allow_pickle=True)
        print("Navigation results saved at ",'nav_result/{}.out'.format(os.path.basename(obs_file)))
        if(AMB_FIX):
            np.save('nav_result/{}.fixout'.format(os.path.basename(obs_file)),Out_log_fix,allow_pickle=True)
            print("Navigation results (AR) saved at ",out_path+'/{}.fixout'.format(os.path.basename(obs_file)))
    
    return True

In [33]:
#读取yaml
PTK_YAML="xmls/Easy4PTK.yaml"
with open(PTK_YAML,"r") as f:
    cfg=yaml.safe_load(f)
    print("sys_index set as multi-GNSS")
    PPP_YAML_GCE(cfg)

sys_index set as multi-GNSS
Easy4PPP Configurations:
obs_file data/OBS/zim21540.25o
sys_indexs ['G', 'C', 'E']
obs_type [['C1C', 'L1C', 'D1C', 'S1C', 'C2W', 'L2W', 'D2W', 'S2W'], ['C2I', 'L2I', 'D2I', 'S2I', 'C6I', 'L6I', 'D6I', 'S6I'], ['C1C', 'L1C', 'D1C', 'S1C', 'C5Q', 'L5Q', 'D5Q', 'S5Q']]
freqs [[1575420000.0, 1227600000.0], [1561098000.0, 1268520000.0], [1575420000.0, 1176450000.0]]
SP3_file data/Peph_clk/20251540/cnt23692.sp3
CLK_file data/Peph_clk/20251540/cnt23692.clk
ATX_file data/ATX/igs20.atx
out_path test
ion_param []
BRDC_file data/brdc/brdc1540.25p
dcb_correction 0
dcb_products CAS
dcb_file_0 data/DCB/CAS0MGXRAP_20251540000_01D_01D_DCB.BSX
dcb_file_1 
dcb_file_2 
obs_start 480
obs_epoch 2400
out_age 31
dy_mode dynamic
el_threthod 10.0
ex_threshold_v 30
ex_threshold_v_sigma 4
Mw_threshold 2.5
GF_threshold 0.15
sat_out ['C01', 'C02', 'C03', 'C04', 'C05', 'C06', 'C07', 'C08', 'C09', 'C10', 'C11', 'C12', 'C13', 'C14', 'C15', 'C16', 'C17', 'C18']
AMB_FIX 0
OSB_path data/Peph_

100%|██████████| 2400/2400 [04:07<00:00,  9.69it/s]


Navigation results saved at  test/zim21540.25o.out
